# V3 Colab setup and freeze guard

Mount Drive, clone the pinned repository, copy the user-uploaded frozen V3 package to local Colab storage, and validate every checksum before any training.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/kltn')
PINNED_COMMIT = 'bbe308c'  # update only when deliberately locking a new code commit
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/maiphuowng205/kltn.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', PINNED_COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', PINNED_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-colab.txt')], check=True)
V3_DRIVE_ROOT = Path('/content/drive/MyDrive/kltn/frozen/vn_v3_lseg_2026-08-03')
WORKSPACE = Path('/content/vn_v3_workspace')
DATA_ROOT = WORKSPACE / 'data' / 'lseg_v3'
if not (V3_DRIVE_ROOT / 'data' / 'lseg_v3').exists() and not (V3_DRIVE_ROOT / 'curated').exists():
    raise FileNotFoundError('Upload the frozen V3 package to V3_DRIVE_ROOT before continuing.')
if (V3_DRIVE_ROOT / 'data' / 'lseg_v3').exists():
    if (WORKSPACE / 'data').exists(): shutil.rmtree(WORKSPACE / 'data')
    shutil.copytree(V3_DRIVE_ROOT / 'data', WORKSPACE / 'data')
else:
    if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(V3_DRIVE_ROOT, DATA_ROOT)
print({'repo': str(REPO), 'commit': subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip(), 'data_root': str(DATA_ROOT)})

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(REPO / 'scripts' / 'validate_v3_contract.py'), '--workspace-root', str(WORKSPACE)], check=True)